# panjika

> the register of deeds

A panji is the genealogical register a panjikar keeps: who descends from whom, appended and
never rewritten. This is the same thing for agent work. Every session, every file it touched,
and where that change ended up in git, in one append-only JSONL ledger that every harness in a
repository writes to.

Agent sessions are logged today. What is missing is the join: **which session changed this file,
in what context, and did the change survive.**

In [ ]:
#| hide
from panjika.core import *
from panjika.read import *
from panjika.write import *
from panjika.git import *
from panjika.harness import *

## Three harnesses, one repository

The setup below is a billing module, and three agent sessions a week apart. Claude Code arrives
through its hooks, Codex and Ramabana through the writer directly. None of them is told where
the ledger is: `find_home` walks up from the working directory each was given.

In [ ]:
#| hide
import subprocess, tempfile, time
from pathlib import Path

def git(root, *a): subprocess.run(['git', *a], cwd=root, capture_output=True, check=True)

d = Path(tempfile.mkdtemp())/'billing'; d.mkdir(parents=True)
git(d, 'init', '-q', '-b', 'main')
git(d, 'config', 'user.email', 'sam@example.com'); git(d, 'config', 'user.name', 'Sam')
(d/'charges.py').write_text('def total(rows):\n    return sum(r["amount"] for r in rows)\n')
(d/'refunds.py').write_text('def refundable(row):\n    return row["amount"] > 0\n')
git(d, 'add', '-A'); git(d, 'commit', '-qm', 'the billing module')
Home(d/'.panjika').init()

In [ ]:
def cc(**kw): ingest({'session_id': 'cc-4f21', 'cwd': str(d), **kw}, 'claude-code', d/'.panjika')

cc(hook_event_name='SessionStart', model='opus-5')
cc(hook_event_name='UserPromptSubmit',
   user_input='total() blows up on refunds. Make it skip negative amounts.')
cc(hook_event_name='PostToolUse', tool_name='Read',
   tool_input={'file_path': str(d/'charges.py')}, tool_response='ok')
(d/'charges.py').write_text(
    'def total(rows):\n    return sum(r["amount"] for r in rows if r["amount"] > 0)\n')
cc(hook_event_name='PostToolUse', tool_name='Edit',
   tool_input={'file_path': str(d/'charges.py')}, tool_response='ok')
cc(hook_event_name='PostToolUseFailure', tool_name='Bash',
   tool_input={'command': 'pytest -q'}, tool_response='1 failed')
cc(hook_event_name='SessionEnd', reason='clear')

git(d, 'commit', '-aqm', 'skip negative amounts in total()')
link_commit('HEAD', home=d/'.panjika', start=d)

Wednesday. Ramabana rounds the total, and a person disagrees an hour later.

In [ ]:
time.sleep(1.1)
sc = Scribe(home=d/'.panjika', session='rb-0c7e', start=d)
sc.begin('ramabana', model='sonnet', prompt='total() should round to 2dp')
(d/'charges.py').write_text(
    'def total(rows):\n    return round(sum(r["amount"] for r in rows if r["amount"] > 0), 2)\n')
sc.touch(d/'charges.py', 'edit', sc.step('edit_file', target='charges.py', secs=0.5))
sc.end('done')

time.sleep(1.1)
(d/'charges.py').write_text(
    'def total(rows):\n    # amounts are already in cents; rounding here loses a penny per invoice\n'
    '    return sum(r["amount"] for r in rows if r["amount"] > 0)\n')
git(d, 'commit', '-aqm', 'no rounding: amounts are already in cents')

## The trail over a file

Every session and every commit that ever touched one path, on one timeline. This is the question
the package exists for.

In [ ]:
for r in blend('charges.py', home=d/'.panjika', start=d):
    who = r.get('short') if r.kind == 'commit' else f"{r.session} {r.harness}"
    print(f"{r.kind:<8} {who:<24} {r.title[:60]}")

## Did it land

`landed` matches the exact lines a session wrote against `git blame`. The Monday session is in
the history. The Wednesday one is not, and the verdict names the commit that took its place.

In [ ]:
for v in landed('cc-4f21', home=d/'.panjika', start=d): print(v.line())
for v in landed('rb-0c7e', home=d/'.panjika', start=d): print(v.line())

An agent that is about to round the total again reads that and stops. The `why` is
the point: it names the commit, so the next thing to read is obvious.

`landed()` with no arguments answers for the newest session in the repository, which is what an
agent picking work back up wants first.

## The log

`sessions` filters compose, and the counters are computed from the records rather than written,
because a hook fires once per tool call in its own process and cannot keep a running total.

In [ ]:
led = Ledger(d/'.panjika')
for r in led.sessions():
    row = led.session(r.session)
    print(f"{r.session}  {r.harness}/{r.model}  {row.n_steps} steps, {row.steps_fail} failed")
    print(f"    {r.prompt[:70]}")
    print(f"    {', '.join(t.path for t in row.files)}")

## How it is stored

Two tiers of JSONL under `.panjika/`. `ledger/` is small, carries no source, and is meant to be
committed. `detail/` is machine-local and holds what a summary cannot: whole tool arguments,
whole outputs, and the hashes of the lines each change added, which are what make `landed`
exact.

In [ ]:
print((d/'.panjika'/'.gitattributes').read_text())
print(Ledger(d/'.panjika').stats())

Nothing is ever rewritten. A session is not one record; it is every record that named
it, folded in time order. `merge=union` above is the other half: two branches that both appended
merge to the union of their lines instead of a conflict, and every reader deduplicates by record
id, which is what makes taking both sides safe.

## Setting it up

```sh
pip install panjika
panjika install       # Claude Code hooks, a git post-commit hook, and the ledger folder
panjika trail FILE    # who has touched this
panjika landed        # what became of what I wrote
```

Adding a harness is a function that turns its payload into a list of calls, and a line in
`ADAPTERS`. Adapters write nothing, so a test for one needs no disk.

The Codex adapter is **not verified**: its hook payload was not documented anywhere reachable
when this was written. It reads several spellings of every field so a near miss still records
something, and `panjika install` prints its configuration rather than writing it.